# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates the loading and exploration of the FAIR² dataset package using the `mlcroissant` library.

### Dataset Source
The dataset is accessible via its Croissant schema URL, enabling programmatic loading and inspection of metadata and data tables.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset Name: ", metadata.name)
print("Description: ", metadata.description)

## 2. Data Overview
Review available record sets, their fields, and accessible `@id`s.

The primary interface for identifying data resources in a Croissant dataset is through each Record Set's `@id`.

List all record sets and display their fields with their `@id`s.

In [ ]:
# List all record sets, referencing by @id
record_sets = [r['@id'] for r in dataset.metadata.record_sets]
print("Available Record Sets (@id):")
for rs in dataset.metadata.record_sets:
    print(f"- {rs['@id']} : {rs['name'] if 'name' in rs else '(no name)'}")

# Display the fields (columns) in each record set, also by their @id
for rs in dataset.metadata.record_sets:
    print(f"\nRecord Set @id: {rs['@id']}")
    if 'fields' in rs:
        print("Fields:")
        for f in rs['fields']:
            fname = f['name'] if 'name' in f else ''
            print(f"  - {f['@id']} : {fname}")
    else:
        print("  (No fields listed)")

## 3. Data Extraction
Load data from each Record Set into a DataFrame for further analysis.

- Each Record Set must be referenced by its `@id`.
- Data columns will use their `@id`s for access.
- Since record sets may be large, we'll load as DataFrames and show columns and example records.

In [ ]:
# Extract data from each record set by @id
# Use variables to store mappings from @id to DataFrames
dataframes = {}
for rs in record_sets:
    try:
        records = list(dataset.records(record_set=rs))
        if records:
            dataframes[rs] = pd.DataFrame(records)
            print(f"\nLoaded DataFrame for Record Set @id: {rs}")
            print(f"Columns (@id): {list(dataframes[rs].columns)}")
            display(dataframes[rs].head())
        else:
            print(f"\nNo records found for Record Set @id: {rs}")
    except Exception as e:
        print(f"\nError loading records for Record Set @id: {rs}: {e}")

## 4. Exploratory Data Analysis (EDA)
Explore numeric fields, filter by threshold, normalize, and group the data for preliminary insight.

All fields and operations use `@id` references for columns.

In [ ]:
# Automatically select a numeric field from a record set (using @id)
# For demonstration, select the first numeric field found in the first DataFrame
import numpy as np

selected_rs = None
numeric_field_id = None
group_field_id = None

for rsid, df in dataframes.items():
    for c in df.columns:
        # Look for an int/float column
        if np.issubdtype(df[c].dtype, np.number):
            numeric_field_id = c
            selected_rs = rsid
            break
    if selected_rs:
        break

if selected_rs and numeric_field_id:
    print(f"Selected Record Set @id: {selected_rs}")
    print(f"Selected Numeric Field @id: {numeric_field_id}")
    # Choose a (different) categorical field for grouping, if available
    df = dataframes[selected_rs]
    possible_group_fields = [c for c in df.columns if c != numeric_field_id and (df[c].dtype == 'O')]
    if possible_group_fields:
        group_field_id = possible_group_fields[0]
        print(f"Selected Group Field @id: {group_field_id}")

    # Set threshold as the mean (or 10 if mean is not available)
    threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # If a grouping field is available, group and aggregate
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric fields found to analyze.")

## 5. Visualization
Visualize the distribution of the chosen numeric field and its relationship to the grouping field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rs and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[selected_rs][numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[selected_rs])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field selected for visualization.")

## 6. Conclusion
We have successfully explored the FAIR² dataset using the Croissant schema and `mlcroissant`. All references were made using entity `@id`s to identify record sets and fields. This approach ensures robust and reproducible data access across schema versions.

- To conduct further or domain-specific analysis, use the record set and field `@id`s displayed above.
- The dataset provides ordered logistic regression outputs for knowledge adoption in rangeland management in Northern Kenya.
- Further steps might involve more advanced modeling or cross-referencing with domain knowledge.